# Spoof-detector separability notebook

Ingest a folder of `amispoof-session-*.json` dumps produced by the
amispoof demo's **↓ Report** button after the Phase F frame-log
instrumentation. For every numeric feature exposed by the 25 analyzers we:

1. Build a long-form DataFrame `frame × feature` with the `capture_label` baked in.
2. Compute per-feature **AUC** and **d′** LIVE vs REPLAY (the only two
   metrics worth quoting in a paper for binary class separability).
3. Rank features, plot the top 10 distributions, draw a correlation heatmap.
4. Fit a baseline **logistic regression** with 5-fold CV (leave-one-session-out
   when there are enough sessions). The CV AUC is the floor every fancier model
   has to beat — if LogReg already hits 0.95+, ship it; no need for XGBoost.
5. If LogReg plateaus, escalate to **XGBoost** — still tabular, ONNX-exportable,
   runs in the browser via `onnxruntime-web`.

**Inputs:** drop session JSONs into `./data/` (or set `SESSIONS_DIR` below).
Each file's `environment.capture_label` is its ground-truth class.

**Outputs:** ranked-feature table, top-feature plots, correlation heatmap,
CV-AUC for LogReg (and XGBoost if installed). All artefacts re-render from the
JSONs every run — there is no hidden state between cells.

In [ ]:
# === Setup ===
from __future__ import annotations
import json, math, os, glob
from pathlib import Path
import numpy as np
import pandas as pd

# Look first at sibling docs/ (where the existing handful of dumps live),
# then at a dedicated ./data/ folder once the dataset grows. Override here
# if you keep dumps somewhere else.
REPO_ROOT = Path('..').resolve()
CANDIDATE_DIRS = [REPO_ROOT / 'notebooks' / 'data', REPO_ROOT / 'docs', REPO_ROOT / 'data']
SESSIONS_DIR = next((d for d in CANDIDATE_DIRS if d.exists() and any(d.glob('amispoof-session-*.json'))), None)
if SESSIONS_DIR is None:
    raise SystemExit(f'No session JSONs found. Drop dumps into one of: {[str(d) for d in CANDIDATE_DIRS]}')
print('Loading sessions from:', SESSIONS_DIR)
session_paths = sorted(SESSIONS_DIR.glob('amispoof-session-*.json'))
print(f'  found {len(session_paths)} files')

In [ ]:
# === Load + flatten ===
# Each session JSON has a `frame_log` array (added by the Phase F instrumentation).
# We unroll every frame into one DataFrame row carrying the session id + class label,
# then pull every numeric detail value out of `analyzer_scores[*].details` as its own
# column. Missing keys (analyzer didn't fire on that frame) become NaN — sklearn
# handles those after the median-impute below.

def flatten_frame(frame_row: dict, analyzer_name: str, payload: dict | None) -> dict:
    out = {}
    if not payload:
        return out
    if 'score' in payload:
        out[f'{analyzer_name}.score'] = payload['score']
    details = payload.get('details') or {}
    for k, v in details.items():
        if isinstance(v, (int, float, bool)):
            out[f'{analyzer_name}.{k}'] = float(v)
        # Strings and nested objects (top_active, per_blendshape) are skipped
        # for the baseline pass; a follow-up cell can pivot them on demand.
    return out

rows = []
for path in session_paths:
    sess = json.loads(path.read_text(encoding='utf-8'))
    env = sess.get('environment') or {}
    label = (env.get('capture_label') or 'UNLABELED').upper()
    # Class collapse for the binary classifier. REPLAY_* / PRINT / MASK / DEEPFAKE
    # all roll up into SPOOF. UNLABELED rows are kept for inspection but excluded
    # from any modelling cell.
    if label == 'LIVE':
        binary = 'LIVE'
    elif label.startswith('REPLAY') or label in ('PRINT', 'MASK', 'DEEPFAKE'):
        binary = 'SPOOF'
    else:
        binary = 'UNLABELED'
    frame_log = sess.get('frame_log') or []
    if not frame_log:
        # Schema v1 dumps (pre-Phase-F) have no frame_log — still useful as a
        # one-point session-level row. Synthesize a single row from latest_*.
        frame_log = [{
            't_sec': sess.get('verdict', {}).get('session_duration_sec', 0),
            'frame_id': sess.get('verdict', {}).get('frames_analyzed', 0),
            'analyzer_scores': sess.get('latest_analyzer_scores') or {},
            'confidence': sess.get('verdict', {}).get('confidence', 0),
        }]
    for f in frame_log:
        flat = {
            'session': path.stem,
            'class': binary,
            'class_fine': label,
            'ambient': env.get('ambient_label'),
            'replay_device': env.get('replay_device'),
            't_sec': f.get('t_sec'),
            'frame_id': f.get('frame_id'),
            'confidence': f.get('confidence'),
            'fps': f.get('fps'),
        }
        for analyzer_name, payload in (f.get('analyzer_scores') or {}).items():
            flat.update(flatten_frame(f, analyzer_name, payload))
        rows.append(flat)

frames = pd.DataFrame(rows)
print(f'frames.shape = {frames.shape}')
print('class counts:')
print(frames['class'].value_counts())
print('\nper-session frame count (first 10):')
print(frames.groupby(['session', 'class']).size().head(10))

In [ ]:
# === Per-feature separability: AUC + d-prime ===
# Only features non-null in at least 30% of LIVE *and* 30% of SPOOF frames are
# scored — a feature that fires only on one class is unreliable evidence (the
# AUC inflates because the comparison is degenerate, not because the feature
# discriminates).
from sklearn.metrics import roc_auc_score

labelled = frames[frames['class'].isin(['LIVE', 'SPOOF'])].copy()
if labelled.empty:
    raise SystemExit('No labelled frames yet — capture LIVE + SPOOF sessions first.')
y = (labelled['class'] == 'SPOOF').astype(int).values

feature_cols = [c for c in labelled.columns if c not in {
    'session', 'class', 'class_fine', 'ambient', 'replay_device',
    't_sec', 'frame_id', 'confidence', 'fps',
}]

def safe_auc(y_true, x):
    mask = ~np.isnan(x)
    if mask.sum() < 30:
        return np.nan
    # roc_auc_score is direction-agnostic if we report max(auc, 1-auc).
    try:
        a = roc_auc_score(y_true[mask], x[mask])
    except ValueError:
        return np.nan
    return max(a, 1 - a)

def d_prime(x_live, x_spoof):
    if len(x_live) < 5 or len(x_spoof) < 5:
        return np.nan
    mu_l, mu_s = np.nanmean(x_live), np.nanmean(x_spoof)
    s = math.sqrt(0.5 * (np.nanvar(x_live) + np.nanvar(x_spoof)))
    if s < 1e-9:
        return np.nan
    return abs(mu_l - mu_s) / s

rows = []
for col in feature_cols:
    x = labelled[col].values.astype(float)
    live_x = labelled.loc[labelled['class'] == 'LIVE', col].dropna().values
    spoof_x = labelled.loc[labelled['class'] == 'SPOOF', col].dropna().values
    coverage_live = len(live_x) / max(1, (labelled['class'] == 'LIVE').sum())
    coverage_spoof = len(spoof_x) / max(1, (labelled['class'] == 'SPOOF').sum())
    if coverage_live < 0.3 or coverage_spoof < 0.3:
        continue
    rows.append({
        'feature': col,
        'auc': safe_auc(y, x),
        'd_prime': d_prime(live_x, spoof_x),
        'mean_live': float(np.nanmean(live_x)),
        'mean_spoof': float(np.nanmean(spoof_x)),
        'std_live': float(np.nanstd(live_x)),
        'std_spoof': float(np.nanstd(spoof_x)),
        'coverage_live': coverage_live,
        'coverage_spoof': coverage_spoof,
    })

ranking = pd.DataFrame(rows).sort_values('auc', ascending=False).reset_index(drop=True)
print('Top 20 features by AUC:')
ranking.head(20)

In [ ]:
# === Distribution plots for the top features ===
import matplotlib.pyplot as plt

top_n = 8
top_features = ranking.head(top_n)['feature'].tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flat, top_features):
    live_x = labelled.loc[labelled['class'] == 'LIVE', col].dropna()
    spoof_x = labelled.loc[labelled['class'] == 'SPOOF', col].dropna()
    if live_x.empty or spoof_x.empty:
        ax.set_title(f'{col} (no data)')
        continue
    ax.hist(live_x, bins=30, alpha=0.55, label='LIVE', density=True)
    ax.hist(spoof_x, bins=30, alpha=0.55, label='SPOOF', density=True)
    auc = ranking.loc[ranking['feature'] == col, 'auc'].iloc[0]
    ax.set_title(f'{col}\nAUC={auc:.3f}', fontsize=9)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# === Correlation heatmap (top 20 features) ===
# Two highly-correlated features carry roughly the same information — useful
# to know before you stack them into a logistic regression (multicollinearity
# inflates coefficient variance).
import seaborn as sns

top_for_corr = ranking.head(20)['feature'].tolist()
corr = labelled[top_for_corr].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Pearson correlation — top 20 features by AUC')
plt.tight_layout()
plt.show()

In [ ]:
# === Baseline logistic regression ===
# Group-aware split: every fold's test set draws from sessions the train set
# never saw, so we don't claim AUC the model only got from memorising a
# subject's idiosyncratic noise. GroupKFold is the right primitive here.
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score

model_features = ranking.head(20)['feature'].tolist()
X = labelled[model_features].values.astype(float)
y = (labelled['class'] == 'SPOOF').astype(int).values
groups = labelled['session'].values

n_groups = len(set(groups))
n_splits = max(2, min(5, n_groups))
if n_groups < 2:
    print(f'Only {n_groups} session(s) loaded — group CV needs at least 2. Skipping.')
else:
    pipe = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler()),
        ('lr', LogisticRegression(max_iter=2000, class_weight='balanced')),
    ])
    gkf = GroupKFold(n_splits=n_splits)
    aucs = cross_val_score(pipe, X, y, groups=groups, scoring='roc_auc', cv=gkf)
    print(f'Logistic regression — {n_splits}-fold group CV AUC: {aucs.mean():.3f} ± {aucs.std():.3f}')
    print(f'  per-fold: {[round(a, 3) for a in aucs]}')
    # Fit-on-all for coefficient inspection (NOT a test score — don't quote this).
    pipe.fit(X, y)
    coefs = pd.Series(pipe.named_steps['lr'].coef_[0], index=model_features).sort_values(key=abs, ascending=False)
    print('\nLogReg coefficients (sign → direction of SPOOF evidence):')
    print(coefs.head(15))

In [ ]:
# === Optional escalation: XGBoost ===
# Skipped automatically if xgboost isn't installed. Run with the same
# group-CV setup so the headline number is directly comparable to LogReg.
try:
    from xgboost import XGBClassifier
    have_xgb = True
except ImportError:
    have_xgb = False
    print('xgboost not installed — skipping. `pip install xgboost` to enable.')

if have_xgb and n_groups >= 2:
    pipe_xgb = Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('xgb', XGBClassifier(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
            use_label_encoder=False, verbosity=0,
        )),
    ])
    aucs_xgb = cross_val_score(pipe_xgb, X, y, groups=groups, scoring='roc_auc', cv=gkf)
    print(f'XGBoost — {n_splits}-fold group CV AUC: {aucs_xgb.mean():.3f} ± {aucs_xgb.std():.3f}')
    print(f'  per-fold: {[round(a, 3) for a in aucs_xgb]}')
    pipe_xgb.fit(X, y)
    importances = pd.Series(
        pipe_xgb.named_steps['xgb'].feature_importances_, index=model_features
    ).sort_values(ascending=False)
    print('\nXGBoost feature importance (gain):')
    print(importances.head(15))

## How to read the output

- **Per-feature AUC table**: anything `>= 0.95` is a single-feature threshold rule
  candidate. Anything `< 0.55` is noise — either drop it from fusion or down-weight.
- **d′**: distance between the LIVE and SPOOF means in pooled-std units. > 1.5 means
  the distributions are visibly separated; > 3 means they barely overlap.
- **Top-feature histograms**: visual sanity check the AUC number isn't an artefact
  of a single outlier session. If both classes have the same distribution shape and
  one is just shifted, a single threshold works; if they're tangled, you need the
  classifier.
- **Correlation heatmap**: red blocks of correlated features = redundant evidence;
  prune to one per block before quoting feature counts in the paper.
- **LogReg CV AUC**: the *honest* number. If > 0.95 — ship LogReg, no further model
  work needed. If 0.85–0.95 — try XGBoost. If < 0.85 — you don't have a model
  problem, you have a *data* problem: collect more sessions with more class diversity
  (different rooms, different replay devices, different subjects) before retraining.